<a href="https://colab.research.google.com/github/Logapriya1911002/Form-Correctness-Detection-using-Pose-Estimation_Multiple-Exercise/blob/main/Form_Correctness_Detection_using_Pose_Estimation_Multiple_Exercise.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [112]:
import sys
print(sys.version)

3.12.12 (main, Oct 10 2025, 08:52:57) [GCC 11.4.0]


In [113]:
!pip install tensorflow opencv-python numpy

In [66]:
import tensorflow as tf
import numpy as np
import cv2


In [67]:
!pip install tensorflow-hub


In [68]:
import tensorflow_hub as hub
import tensorflow as tf


In [69]:
model = hub.load("https://tfhub.dev/google/movenet/singlepose/lightning/4")
movenet = model.signatures["serving_default"]



In [70]:
!unzip exercise_videos.zip

Archive:  exercise_videos.zip
replace exercise_videos/biceps2.mp4? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace exercise_videos/biceps1.mp4? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace exercise_videos/pushup.mp4? [y]es, [n]o, [A]ll, [N]one, [r]ename: n
replace exercise_videos/squats.mp4? [y]es, [n]o, [A]ll, [N]one, [r]ename: n


In [71]:
!ls

exercise_videos  exercise_videos.zip  sample_data


In [114]:
import glob

VIDEO_DIR = "/content/exercise_videos/"
video_list = glob.glob(VIDEO_DIR + "*.mp4")

for i, v in enumerate(video_list):
    print(i, v.split("/")[-1])

choice = int(input("Select video index: "))
video_path = video_list[choice]

print("Selected:", video_path)

0 squats.mp4
1 biceps1.mp4
2 biceps2.mp4
3 pushup.mp4
Select video index: 0
Selected: /content/exercise_videos/squats.mp4


In [115]:
def process_video(video_path):

    # ---------- Video read ----------
    cap = cv2.VideoCapture(video_path)

    OUTPUT_WIDTH  = 2160
    OUTPUT_HEIGHT = 3840

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0 or fps is None:
        fps = 25

    raw_output = "raw_output.mp4"

    out = cv2.VideoWriter(
        raw_output,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (OUTPUT_WIDTH, OUTPUT_HEIGHT)
    )

    print("Writer opened:", out.isOpened())

In [ ]:
### video_path = "/content/5319426-uhd_2160_3840_25fps.mp4"

In [116]:
def calculate_angle(a, b, c):
    a = np.array(a)
    b = np.array(b)
    c = np.array(c)

    ba = a - b
    bc = c - b

    cosine = np.dot(ba, bc) / (np.linalg.norm(ba) * np.linalg.norm(bc))
    angle = np.degrees(np.arccos(np.clip(cosine, -1.0, 1.0)))

    return angle


In [117]:
def detect_pose(frame):
    img = cv2.resize(frame, (192, 192))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = tf.convert_to_tensor(img, dtype=tf.int32)
    img = tf.expand_dims(img, axis=0)

    outputs = movenet(img)
    keypoints = outputs['output_0'].numpy()
    return keypoints


In [118]:
def detect_exercise_from_filename(path):
    name = path.lower()
    if "bicep" in name:
        return "bicep_curl"
    elif "squat" in name:
        return "squat"
    elif "pushup" in name:
        return "pushup"
    else:
        return "unknown"


In [119]:
def process_video(video_path):

    # ---------- Video read ----------
    exercise = detect_exercise_from_filename(video_path)
    if exercise == "unknown":
        print("Unknown exercise — skipping")
        return
    cap = cv2.VideoCapture(video_path)

    OUTPUT_WIDTH  = 2160
    OUTPUT_HEIGHT = 3840

    fps = cap.get(cv2.CAP_PROP_FPS)
    if fps == 0 or fps is None:
        fps = 25

    out = cv2.VideoWriter(
        "raw_output.mp4",
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (OUTPUT_WIDTH, OUTPUT_HEIGHT)
    )

    LEFT_SHOULDER = 5
    LEFT_ELBOW    = 7
    LEFT_WRIST   = 9
    LEFT_HIP = 11
    LEFT_KNEE = 13
    LEFT_ANKLE = 15

    counter = 0
    stage = None
    prev_elbow_x = None
    ELBOW_STABILITY_THRESHOLD = 0.05

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        frame = cv2.resize(frame, (OUTPUT_WIDTH, OUTPUT_HEIGHT))
        keypoints = detect_pose(frame)

        shoulder = keypoints[0][0][LEFT_SHOULDER][:2]
        elbow    = keypoints[0][0][LEFT_ELBOW][:2]
        wrist    = keypoints[0][0][LEFT_WRIST][:2]
        hip      = keypoints[0][0][LEFT_HIP][:2]
        knee     = keypoints[0][0][LEFT_KNEE][:2]
        ankle    = keypoints[0][0][LEFT_ANKLE][:2]

        elbow_angle = calculate_angle(shoulder, elbow, wrist)
        back_angle  = calculate_angle(shoulder, hip, knee)
        knee_angle  = calculate_angle(hip, knee, ankle)
        elbow_x = elbow[0]
        if prev_elbow_x is not None:
          elbow_stable = abs(elbow_x - prev_elbow_x) < ELBOW_STABILITY_THRESHOLD
        else:
          elbow_stable = True   # <--- important: define it on first frame
        prev_elbow_x = elbow_x
        if exercise == "bicep_curl":
            if elbow_angle > 150:
                stage = "down"
            elif elbow_angle < 50 and stage == "down":
                stage = "up"
                counter += 1
            correct = elbow_stable and back_angle > 160

        elif exercise == "squat":
            if knee_angle > 160:
                stage = "up"
            elif knee_angle < 90 and stage == "up":
                stage = "down"
                counter += 1
            correct = knee_angle < 100 and back_angle > 150

        elif exercise == "pushup":
            if elbow_angle > 160:
                stage = "up"
            elif elbow_angle < 70 and stage == "up":
                stage = "down"
                counter += 1
            correct = elbow_angle < 90 and back_angle > 160

        color = (0,255,0) if correct else (0,0,255)
        feedback = "Correct Form" if correct else "Incorrect Form"
        cv2.putText(frame, f"{exercise}", (30,40),
                    cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
        cv2.putText(frame, f"Reps: {counter}", (30,90),
                    cv2.FONT_HERSHEY_SIMPLEX, 1.2, (255,255,255), 3)
        cv2.putText(frame, feedback, (30,150),
            cv2.FONT_HERSHEY_SIMPLEX, 1.2, color, 3)

        cv2.putText(frame, f"Elbow Angle: {int(elbow_angle)}", (30,210),
            cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)

        cv2.putText(frame, f"Back Angle: {int(back_angle)}", (30,250),
            cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)

        if exercise in ["squat", "pushup"]:
          cv2.putText(frame, f"Knee Angle: {int(knee_angle)}", (30,290),
                cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)

        out.write(frame)

    cap.release()
    out.release()

    print("Raw video saved.")




In [120]:
process_video(video_path)

Raw video saved.


In [121]:
!ffmpeg -y -i raw_output.mp4 -vcodec libx264 -pix_fmt yuv420p final_output.mp4


ffmpeg version 4.4.2-0ubuntu0.22.04.1 Copyright (c) 2000-2021 the FFmpeg developers
  built with gcc 11 (Ubuntu 11.2.0-19ubuntu1)
  configuration: --prefix=/usr --extra-version=0ubuntu0.22.04.1 --toolchain=hardened --libdir=/usr/lib/x86_64-linux-gnu --incdir=/usr/include/x86_64-linux-gnu --arch=amd64 --enable-gpl --disable-stripping --enable-gnutls --enable-ladspa --enable-libaom --enable-libass --enable-libbluray --enable-libbs2b --enable-libcaca --enable-libcdio --enable-libcodec2 --enable-libdav1d --enable-libflite --enable-libfontconfig --enable-libfreetype --enable-libfribidi --enable-libgme --enable-libgsm --enable-libjack --enable-libmp3lame --enable-libmysofa --enable-libopenjpeg --enable-libopenmpt --enable-libopus --enable-libpulse --enable-librabbitmq --enable-librubberband --enable-libshine --enable-libsnappy --enable-libsoxr --enable-libspeex --enable-libsrt --enable-libssh --enable-libtheora --enable-libtwolame --enable-libvidstab --enable-libvorbis --enable-libvpx --enab

In [122]:
from google.colab import files
files.download("final_output.mp4")


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>